# HCDE 530 — Week 5: App reviews exploration

This notebook loads **`app_reviews_demo.csv`** and walks through five questions about the data. Run each cell with **Shift + Enter** and read the output before moving on.

---

## Setup

Import pandas and load the CSV. The working directory should be the folder that contains `app_reviews_demo.csv` (for example, open the notebook from **`HCDE 530 Week 5 Project`**).

In [2]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('app_reviews_demo.csv', encoding='utf-8')
print('pandas version:', pd.__version__)
print('rows, columns:', df.shape)

pandas version: 2.3.3
rows, columns: (500, 10)


---
## 1. What does your dataset look like? `head()`, `info()`

**`head()`** shows column names and a sample of rows so you can sanity-check values and formatting. **`info()`** lists each column’s dtype, how many non-null values there are, and memory use — your map of what is in the table.

In [3]:
df.head(10)

,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3
5,6,Lookback,user research,5,The guest access feature is perfect for extern...,2024-11-08,14,False,NaN,5.3.1
6,7,Fieldkit,field research,5,Project folders keep multi-phase studies manag...,2023-06-16,23,True,mobile,3.0.0
7,8,Maze,usability testing,5,I was running my first study within 20 minutes...,2023-06-25,34,False,NaN,4.1.0
8,9,Dovetail,research repository,4,The guest access feature is perfect for extern...,2023-08-23,2,False,desktop,2.0.0
9,10,Miro,collaborative whiteboard,3,"Works fine — nothing remarkable, nothing broken.",2024-08-03,45,True,NaN,9.4.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 500 non-null    int64 
 1   app                500 non-null    object
 2   category           500 non-null    object
 3   rating             500 non-null    int64 
 4   review             500 non-null    object
 5   date               500 non-null    object
 6   helpful_votes      500 non-null    int64 
 7   verified_purchase  500 non-null    bool  
 8   device_type        437 non-null    object
 9   app_version        389 non-null    object
dtypes: bool(1), int64(3), object(6)
memory usage: 35.8+ KB


---
## 2. What’s the distribution of your most important column?

For app reviews, **`rating`** (1–5) is the main outcome: it tells you how satisfied reviewers are. **`value_counts()`** shows how often each rating appears; sorting by the index puts stars in order.

In [5]:
rating_counts = df['rating'].value_counts().sort_index()
rating_counts

rating
1     29
2     43
3     61
4    160
5    207
Name: count, dtype: int64

In [6]:
rating_pct = df['rating'].value_counts(normalize=True).sort_index().mul(100).round(1)
rating_pct.astype(str) + '%'

rating
1     5.8%
2     8.6%
3    12.2%
4    32.0%
5    41.4%
Name: proportion, dtype: object

---
## 3. Filter to a meaningful subset. What’s in it?

Here we keep only **low ratings (1 or 2)** — the reviews that usually signal problems worth investigating. That is a **meaningful subset** because it separates critical feedback from neutral or positive noise.

We store the result in **`low_ratings`** and inspect shape and a few rows.

In [7]:
low_ratings = df[df['rating'] <= 2].copy()
print('Subset size:', low_ratings.shape[0], 'rows out of', len(df))
low_ratings.head(10)

Subset size: 72 rows out of 500


,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
26,27,Fieldkit,field research,2,No built-in way to generate a structured debri...,2024-04-04,8,True,mobile,2.5.0
31,32,Maze,usability testing,1,Session sharing links occasionally expire befo...,2024-06-27,10,True,tablet,4.2.3
32,33,Lookback,user research,2,Loading large projects takes noticeably longer...,2024-11-22,15,True,desktop,5.2.0
42,43,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-19,9,True,mobile,NaN
51,52,Maze,usability testing,1,Session sharing links occasionally expire befo...,2024-09-12,32,True,mobile,NaN
74,75,Miro,collaborative whiteboard,1,I've lost tags twice after a session due to a ...,2023-12-25,14,True,desktop,NaN
79,80,Dovetail,research repository,2,The Figma integration is read-only; I can't pu...,2024-08-02,2,True,desktop,1.8.4
102,103,Miro,collaborative whiteboard,2,Storage limits hit quickly when you're recordi...,2024-09-10,16,True,desktop,NaN


In [8]:
low_ratings[['app', 'category', 'rating', 'review']].head(15)

,app,category,rating,review
0,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...
1,Fieldkit,field research,2,Search results are slow when the repository is...
26,Fieldkit,field research,2,No built-in way to generate a structured debri...
31,Maze,usability testing,1,Session sharing links occasionally expire befo...
32,Lookback,user research,2,Loading large projects takes noticeably longer...
42,Fieldkit,field research,2,Search results are slow when the repository is...
51,Maze,usability testing,1,Session sharing links occasionally expire befo...
74,Miro,collaborative whiteboard,1,I've lost tags twice after a session due to a ...
79,Dovetail,research repository,2,The Figma integration is read-only; I can't pu...
102,Miro,collaborative whiteboard,2,Storage limits hit quickly when you're recordi...


---
## 4. Group by a category and find the average of a numeric column

We group by **`category`** (the type of product) and take the **mean `rating`** per category. That answers: *which kinds of apps tend to score higher or lower in this sample?*

In [9]:
df.groupby('category')['rating'].mean().round(2).sort_values(ascending=False)

category
research repository         4.12
collaborative whiteboard    4.02
usability testing           4.00
user research               3.90
field research              3.67
Name: rating, dtype: float64

In [10]:
df.groupby('category')['rating'].agg(['mean', 'count', 'std']).round(2)

,mean,count,std
category,,,
collaborative whiteboard,4.02,121,1.20
field research,3.67,92,1.29
research repository,4.12,89,1.03
usability testing,4.00,93,1.16
user research,3.90,105,1.18


---
## 5. Where are the missing values? Are any columns incomplete?

**`isnull().sum()`** counts missing cells per column. Dividing by row count gives the **share missing**. Any column with a count above zero is **incomplete** for at least some rows — you then decide whether to impute, drop, or analyze missingness separately.

In [11]:
missing = df.isnull().sum()
missing

id                     0
app                    0
category               0
rating                 0
review                 0
date                   0
helpful_votes          0
verified_purchase      0
device_type           63
app_version          111
dtype: int64

In [12]:
pct_missing = df.isnull().mean().mul(100).round(1)
pct_missing[pct_missing > 0]

device_type    12.6
app_version    22.2
dtype: float64

In [13]:
incomplete = missing[missing > 0]
if incomplete.empty:
    print('No missing values in any column.')
else:
    print('Columns with at least one missing value:')
    print(incomplete.to_string())

Columns with at least one missing value:
device_type     63
app_version    111
